In [33]:
def safe_get(data:dict,key:str):
    try:
        return data[key]
    except KeyError:
        return '默认值'
if __name__ == '__main__':
    user = {'name':'Alice','age':25}
    result1 = safe_get(user,'name')
    print(result1)
    result2 = safe_get(user,'age')
    print(result2)
    result3 = safe_get(user,'ket')
    print(result3)

Alice
25
默认值


In [ ]:
import time
import random
from functools import wraps
def retry(max_attempts:int=3,delay:float=0.5):
    def decorator(func):
        @wraps(func)
        def wrapper(*args,**kwargs):
            last_exception = None
            for attempt in range(1,max_attempts+1):
                try:
                    return func(*args,**kwargs)
                except Exception as e:
                    last_exception = e
                    if attempt < max_attempts:
                        print(f"  ⚠️ 第 {attempt} 次失败: {e},{delay}s 后重试...")
                        time.sleep(delay)
                    else:
                        print(f"  ❌ 第 {attempt} 次失败: {e},已达最大重试次数")
            raise last_exception
        return wrapper
    return decorator
@retry(max_attempts=5,delay=0.3)
def unstable_api():
    if random.random() < 0.6:
        raise ConnectionError("服务器繁忙,请稍后重试")
    return "✅ API 调用成功！"
if __name__ == '__main__':
    print("=== 重试装饰器演示 ===\n")
    try:
        result = unstable_api()
        print(result)
    except ConnectionError as e:
        print(e)

=== 重试装饰器演示 ===

  ⚠️ 第 1 次失败: 服务器繁忙,请稍后重试,0.3s 后重试...
  ⚠️ 第 2 次失败: 服务器繁忙,请稍后重试,0.3s 后重试...
  ⚠️ 第 3 次失败: 服务器繁忙,请稍后重试,0.3s 后重试...
✅ API 调用成功！


In [6]:
import asyncio
import random
import time
async def async_fetch(url:str) -> str:
    delay = random.uniform(0.5,2.0)
    await asyncio.sleep(delay)
    return f"响应: {url} (耗时 {delay:.1f}s)"
async def main():
    urls = [f"https://api.example.com/data/{i}" for i in range(6)]
    print("=== 串行执行 ===")
    start = time.time()
    for url in urls:
        result = await async_fetch(url)
        print(result)
    serial_time = time.time() - start
    print(serial_time)
    print("=== 并发执行 (gather) ===")
    start = time.time()
    results = await asyncio.gather(*[async_fetch(url)for url in urls])
    parallel_time = time.time() - start
    for result in results:
        print(result)
    print(parallel_time)
if __name__ == '__main__':
    await main()

=== 串行执行 ===
响应: https://api.example.com/data/0 (耗时 1.4s)
响应: https://api.example.com/data/1 (耗时 1.2s)
响应: https://api.example.com/data/2 (耗时 0.9s)
响应: https://api.example.com/data/3 (耗时 1.7s)
响应: https://api.example.com/data/4 (耗时 0.7s)
响应: https://api.example.com/data/5 (耗时 1.4s)
7.43541955947876
=== 并发执行 (gather) ===
响应: https://api.example.com/data/0 (耗时 0.6s)
响应: https://api.example.com/data/1 (耗时 1.4s)
响应: https://api.example.com/data/2 (耗时 1.7s)
响应: https://api.example.com/data/3 (耗时 1.5s)
响应: https://api.example.com/data/4 (耗时 1.6s)
响应: https://api.example.com/data/5 (耗时 0.9s)
1.7237160205841064


In [8]:
import threading
import time
balance = 1000
def withdraw_unsafe(amount_per_op:int,num_ops:int):
    global balance
    for _ in range(num_ops):
        current = balance
        time.sleep(0.00001)
        balance = current - amount_per_op
def withdraw_safe(lock:threading.Lock,amount_per_op:int,num_ops:int):
    global balance
    for _ in range(num_ops):
        with lock:
            current = balance
            time.sleep(0.00001)
            balance = current - amount_per_op
if __name__ == '__main__':
    NUM_THREADS = 10
    OPS_PER_THREAD = 100
    EXPECTED = 1000 - NUM_THREADS * OPS_PER_THREAD

    balance = 1000
    print("=== ❌ 不加锁 ===")
    threads = [
        threading.Thread(target=withdraw_unsafe,args=(1,OPS_PER_THREAD))
        for _ in range(NUM_THREADS)
    ]
    for t in threads:
        t.start()
    for t in threads:
        t.join()
    print(f"  期望余额: {EXPECTED}")
    print(f"  实际余额: {balance}")
    print(f"  数据丢失: {EXPECTED - balance}（因为数据竞争）\n")
await main()

=== ❌ 不加锁 ===
  期望余额: 0
  实际余额: 899
  数据丢失: -899（因为数据竞争）

=== 串行执行 ===
响应: https://api.example.com/data/0 (耗时 1.2s)
响应: https://api.example.com/data/1 (耗时 1.4s)
响应: https://api.example.com/data/2 (耗时 1.1s)
响应: https://api.example.com/data/3 (耗时 1.7s)
响应: https://api.example.com/data/4 (耗时 1.1s)
响应: https://api.example.com/data/5 (耗时 1.6s)
8.2537522315979
=== 并发执行 (gather) ===
响应: https://api.example.com/data/0 (耗时 1.7s)
响应: https://api.example.com/data/1 (耗时 1.1s)
响应: https://api.example.com/data/2 (耗时 1.2s)
响应: https://api.example.com/data/3 (耗时 1.8s)
响应: https://api.example.com/data/4 (耗时 0.8s)
响应: https://api.example.com/data/5 (耗时 1.0s)
1.7603378295898438


In [12]:
import asyncio
import time
async def slow_api(task_id:int) -> str:
    await asyncio.sleep(task_id)
    return f'任务{task_id}的结果'
async def main():
    print("=== 异步超时控制（timeout=2.5s）===\n")
    for i in range(1,6):
        try:
            result = await asyncio.wait_for(
                slow_api(i),
                timeout = 2.5
            )
        except asyncio.TimeoutError:
            print(f"  ⏰ 任务 {i}: 超时！（需要 {i}s,超过 2.5s 限制）")
if __name__ == '__main__':
    await main()

=== 异步超时控制（timeout=2.5s）===

  ⏰ 任务 3: 超时！（需要 3s,超过 2.5s 限制）
  ⏰ 任务 4: 超时！（需要 4s,超过 2.5s 限制）
  ⏰ 任务 5: 超时！（需要 5s,超过 2.5s 限制）
